# Setup

In [1]:
#pip install -qU langchain langchain-ollama

In [2]:
import os
from langchain.agents import create_agent
from langchain_google_genai import ChatGoogleGenerativeAI
from pydantic import BaseModel

from pydantic import Field
from pathlib import Path
import json
import re
from typing import Literal
os.environ["GOOGLE_API_KEY"] = "" #include your own api key here

In [3]:
patientState = {
    "age": 19,
    "nationality": "Egyptian",
    "gender": "male",

    "goals": [
        "Manage academic stress",
        "Improve sleep consistency",
        "Build confidence in social situations"
    ],

    "symptoms": [
        {
            "name": "excessive_worry",
            "severity": "moderate",
            "duration": "6 weeks",
            "description": "Frequent worries about academic performance and deadlines"
        },
        {
            "name": "sleep_difficulty",
            "severity": "mild",
            "duration": "3 weeks",
            "description": "Difficulty falling asleep before exams"
        },
        {
            "name": "social_anxiety",
            "severity": "mild",
            "duration": "2 months",
            "description": "Nervousness when speaking in groups or meeting unfamiliar people"
        }
    ],

    "patientHistory": {
        "previousMentalHealthTreatment": "None reported",
        "medicalHistory": "No relevant medical history reported",
        "medicationHistory": "No current medication reported",
        "familyMentalHealthHistory": "Unknown",
        "majorLifeEvents": [
            "Increased academic workload this semester"
        ],
        "copingStrategies": [
            "Listening to music",
            "Talking to a close friend"
        ],
        "socialSupport": "Has a few supportive friends",
        "riskHistory": "Not yet assessed",
    }
}

# Session Planner Agent

In [4]:
class SessionPlan(BaseModel):
    session_goal: str
    possible_interventions: list[str]
    session_objectives: list[str] = Field(default_factory=list)

In [5]:
treatment_plan = """
Client Name: David Miller
DOB: 10/14/1988
Age: 37
Date: 10/06/2026

Diagnosis: (F32.2) Major Depressive Disorder, Single Episode, Severe without Psychotic Features

Clinical Information Gaps: None identified from the supplied data.

Problem 1: Severe Major Depressive Disorder (MDD) with symptoms impacting daily functioning (e.g., work routine, sleep patterns) following a corporate downsizing event 6 months ago.

Goal: Reduce depressive symptoms from severe to mild range (PHQ-9 score from 19 to <7), re-engage in daily work routine, and restore sleep patterns within 12 weeks.

Objective 1: Establish therapeutic alliance and collaboratively develop a treatment plan based on Behavioral Activation (BA).
    Intervention 1-1: Conduct a comprehensive assessment, including symptom severity (e.g., PHQ-9), functional impact, and patient goals. Rationale: Essential for understanding the patient's presentation and tailoring treatment.
    Intervention 1-2: Psychoeducation about depression, its course, and the rationale for Behavioral Activation (BA). Rationale: Provides patient understanding and buy-in for treatment [G1].
    Intervention 1-3: Collaboratively develop a treatment plan, including specific, measurable goals and the use of BA. Rationale: Empowers the patient and ensures treatment aligns with their needs [G1].

Objective 2: Implement Behavioral Activation (BA) to increase engagement in activities linked to improved mood.
    Intervention 2-1: Identify activities that are linked to improved mood and develop a schedule of activities. Rationale: BA focuses on increasing engagement in rewarding or mood-improving activities to reduce avoidance and improve mood [G2, G4].
    Intervention 2-2: Monitor activity scheduling and mood changes. Rationale: Tracking progress helps identify patterns and adjust the plan as needed [G2, G4].
    Intervention 2-3: Address barriers to activity engagement (e.g., lack of motivation, social withdrawal). Rationale: Helps the patient overcome obstacles to implementing the BA plan [G2, G4].

Objective 3: Monitor progress, adjust treatment as needed, and plan for relapse prevention.
    Intervention 3-1: Regularly monitor symptom severity (e.g., PHQ-9) and functional improvement. Rationale: Tracks progress towards goals and informs treatment adjustments [G1].
    Intervention 3-2: Review treatment response and adjust interventions as needed. Rationale: Ensures treatment remains effective and responsive to the patient's needs [G1].
    Intervention 3-3: Discuss relapse prevention strategies, including identifying triggers and early warning signs. Rationale: Prepares the patient for potential future episodes [G1].

Monitoring and Reassessment:
- Monitor PHQ-9 score at regular intervals (e.g., every 2-4 weeks) to track symptom severity.
- Assess functional improvement related to work routine and sleep patterns weekly.
- Review treatment progress and adjust the plan based on response.
- Schedule a follow-up assessment within 2-4 weeks if symptoms are improving to discuss progress and plan next steps [G3].
"""

In [6]:
previous_session_summary = {
    "Session 1": """
    David discussed how losing his job affected his mood, sleep,
    and daily routine. He agreed to track his mood and activities.
    """,
    "Session 2": """
    David reported low motivation and difficulty maintaining a routine.
    He identified walking and preparing breakfast as manageable activities.
    """,
    "Session 3": """
    David completed mood tracking but only tried walking once.
    He struggled to get started in the mornings and felt discouraged.
    The session explored barriers to activity and setting smaller goals.
    """
}

In [7]:
LIBRARY_DIR = Path("Therapy")
if not LIBRARY_DIR.exists():
    LIBRARY_DIR = Path(".")

def clean_text(value, limit=None):
    text = " ".join(value) if isinstance(value, list) else str(value or "")
    text = re.sub(r"\s+", " ", text).strip()
    return text if limit is None or len(text) <= limit else text[:limit - 1].rsplit(" ", 1)[0] + "…"

def load_intervention_records():
    records = []
    for filename in ("ba_interventions.json", "cbt_interventions.json", "ipt_interventions.json"):
        with (LIBRARY_DIR / filename).open(encoding="utf-8") as file:
            library = json.load(file)
        for intervention in library["interventions"]:
            records.append({**intervention, "therapy_id": library["therapy_id"], "therapy_name": library["therapy_name"]})
    return records

PLANNER_CANDIDATE_LIMIT = 10
RETRIEVAL_STOP_WORDS = frozenset({"about", "after", "also", "and", "are", "for", "from", "has", "have", "into", "its", "that", "the", "their", "they", "this", "was", "when", "which", "with", "would"})
INTERVENTION_RECORDS = load_intervention_records()
INTERVENTIONS_BY_ID = {record["intervention_id"]: record for record in INTERVENTION_RECORDS}

def search_text(record):
    fields = ("name", "description", "therapeutic_purpose", "when_to_use", "target_patterns", "prerequisites", "cautions")
    return clean_text([clean_text(record.get(field)) for field in fields]).lower()

def retrieve_interventions(query, limit=10):
    terms = {term for term in re.findall(r"[a-z][a-z-]{2,}", query.lower()) if term not in RETRIEVAL_STOP_WORDS}
    scored = []
    for record in INTERVENTION_RECORDS:
        document = search_text(record)
        score = sum(term in document for term in terms)
        if score:
            scored.append((score, record))
    return [record for _, record in sorted(scored, key=lambda item: item[0], reverse=True)[:limit]]

def planner_candidate(record):
    return {
        "id": record["intervention_id"],
        "therapy": record["therapy_id"],
        "name": record["name"],
        "purpose": clean_text(record.get("therapeutic_purpose"), 220),
        "when_to_use": clean_text(record.get("when_to_use"), 240),
        "cautions": clean_text(record.get("cautions"), 180),
    }

def decision_metadata(record):
    return {
        "id": record["intervention_id"],
        "name": record["name"],
        "purpose": clean_text(record.get("therapeutic_purpose"), 240),
        "when_to_use": clean_text(record.get("when_to_use"), 260),
        "prerequisites": clean_text(record.get("prerequisites"), 180),
        "cautions": clean_text(record.get("cautions"), 220),
    }

def delivery_guidance(record):
    return {
        "purpose": clean_text(record.get("therapeutic_purpose"), 400),
        "prerequisites": clean_text(record.get("prerequisites"), 300),
        "cautions": clean_text(record.get("cautions"), 450),
        "possible_questions": record.get("possible_questions", [])[:3],
    }

In [8]:
#  Guided exploration can be suggested for assessment/formulation work.

In [9]:
Session_planner_agent = create_agent(
    model=ChatGoogleGenerativeAI(
        model="gemma-4-26b-a4b-it",
        temperature=0.6,
        thinking_level="minimal",
    ),
    tools=[],
    response_format=SessionPlan,
    system_prompt="""
    You are a therapy session planner.
    
    Use the treatment plan and previous session summary to determine
    the main goal of the upcoming therapy session.
    
    Output:
    one Concise specific, realistic session goal.
    Suggest up to 4 suitable intervention IDs from RETRIEVED CANDIDATES, guided by the treatment plan.
    You may choose only IDs in RETRIEVED CANDIDATES. Consider their purpose, prerequisites, and cautions.
    Suggestions are optional and may be empty; do not force an intervention.
    Also provide 2-4 clear, ordered session_objectives that serve the session goal.
    Objectives may involve understanding a problem, intervention work, or reviewing progress.
    Keep objectives realistic for one session; do not force a formal intervention.
    """,
)

In [10]:
planner_query = "\n".join([treatment_plan, json.dumps(patientState), json.dumps(previous_session_summary)])
retrieved_candidates = retrieve_interventions(planner_query, limit=PLANNER_CANDIDATE_LIMIT)
planner_candidates = [planner_candidate(record) for record in retrieved_candidates]

result = Session_planner_agent.invoke({
    "messages": [{"role": "user", "content": f"""
Treatment Plan:
{treatment_plan}

Patient Information:
{patientState}

Previous Session Summary:
{previous_session_summary}

RETRIEVED CANDIDATES:
{json.dumps(planner_candidates, ensure_ascii=False)}
"""}]
})
Session_plan = result["structured_response"]
print(Session_plan.model_dump_json(indent=2))

session_objectives = list(Session_plan.session_objectives) or [Session_plan.session_goal]
candidate_ids = {record["intervention_id"] for record in retrieved_candidates}
approved_ids = [
    intervention_id for intervention_id in Session_plan.possible_interventions
    if intervention_id in candidate_ids
][:4]
planner_interventions = {
    intervention_id: INTERVENTIONS_BY_ID[intervention_id]
    for intervention_id in approved_ids
}
approved_intervention_metadata = [decision_metadata(record) for record in planner_interventions.values()]

{
  "session_goal": "Address barriers to activity engagement and refine the activity schedule to increase successful implementation of manageable daily routines.",
  "possible_interventions": [
    "ba_troubleshooting_and_contingency_management",
    "ba_activity_scheduling_and_structuring",
    "cbt_action_plans"
  ],
  "session_objectives": [
    "Review the previous week's mood and activity tracking to identify specific obstacles to walking and morning routines.",
    "Collaboratively work on 'outside-in' activation by breaking down morning tasks into even smaller, more achievable steps.",
    "Develop a concrete, actionable plan for the upcoming week with a specific focus on reducing barriers to morning engagement.",
    "Discuss and validate the client's feelings of discouragement, and re-frame frame the progress as part-of-a-process rather than a realization of failure."
  ]
}


# Decision Agent

In [11]:
Action = Literal[
    "clarify", "explore", "assess",
    "validate", "reflect", "support", "repair",
    "summarize", "formulate", "psychoeducate", "reframe",
    "collaborate", "plan", "answer", "transition", "intervene",
]

class TherapyDecision(BaseModel):
    selected_action: Action = Field(description="The conversational move for this turn.")
    goal: str = Field(default="", description=". a concise sentence, saying the Immediate goal of the next response not what the response should say")
    intervention_id: str | None = Field(default=None, description="Approved intervention being used, if any.")
    intervention_control: Literal["none", "pause", "resume"] = "none"
    objective_disposition: Literal["none", "completed", "deferred", "not_addressed"] = "none"


In [12]:
av_actions = {
    "clarify": "Resolve a material ambiguity or missing detail.",
    "explore": "Understand the patient's experience, patterns, or context.",
    "assess": "Gather targeted symptom, function, progress, or risk information.",
    "validate": "Acknowledge feelings or experience without endorsing every conclusion.",
    "reflect": "Communicate the patient's expressed meaning or emotional experience.",
    "support": "Offer appropriate encouragement or reassurance at the patient's pace.",
    "repair": "Address a misunderstanding, disagreement, or interactional discomfort.",
    "summarize": "Consolidate important information or progress.",
    "formulate": "Offer a tentative, collaborative account of a relevant pattern.",
    "psychoeducate": "Briefly explain a relevant psychological concept.",
    "reframe": "Offer a grounded alternative perspective.",
    "collaborate": "Check preferences, readiness, consent, or agreement.",
    "plan": "Agree on a practical next step without a formal intervention.",
    "answer": "Respond directly to an explicit question or request.",
    "transition": "Move appropriately to a different objective, topic, or phase.",
    "intervene": "Deliver one approved, clinically appropriate intervention.",
}

In [13]:
actions_str = "\n".join([f"- {action}: {desc}" for action, desc in av_actions.items()])

In [14]:
reasoning_agent = create_agent(
    model= ChatGoogleGenerativeAI(
        model="gemma-4-26b-a4b-it",
        temperature=0.6,
        thinking_level="minimal")
    ,
    tools=[],
    response_format=TherapyDecision,
    system_prompt= f"""
You are the therapeutic reasoning component. Choose the most useful
next conversational move, not merely the safest or most familiar one.

Available actions:
{actions_str}

Rules:
- Determine what the patient needs from this particular turn.
- Answer direct questions directly. Do not turn every message into
  a request for more information.
- When sufficient information is available, contribute something useful:
  explain a pattern, offer an alternative perspective, suggest a practical
  step, or help the patient make a decision.
- Treat the full transcript as session memory: use facts and answers already
  given, honor corrections, and do not reopen a question the patient has answered.
- Ask one focused question only when its answer would change what you do next;
  otherwise reflect, summarize, offer a tentative understanding, or move forward.
- Gently examine harsh self-judgments when appropriate.
- Do not force an intervention into ordinary conversation.
- Take initiative when a useful direction is evident.
- Treat safety concerns seriously.
- Follow the session plan when appropriate.

Session Tracking:
- Follow the current phase and objective without forcing an intervention.
- Choose one available conversational action for each turn.
- APPROVED INTERVENTIONS are the only formal interventions available this session.
- Use intervene only when an approved intervention fits, its prerequisites permit it,
  and the patient is ready or has agreed to try it. Otherwise use an ordinary action.
- Set intervention_id only when using, pausing, or resuming an approved intervention.
- Use intervention_control=pause when the active intervention should stop for now;
  pair it with the best ordinary action for this turn.
- Use intervention_control=resume only when the paused intervention is appropriate
  again and the patient has agreed to return to it; then choose intervene and its ID.
- Never resume a paused intervention automatically.
- Use transition only after the current phase exit condition is met. It advances
  CHECK_IN_REVIEW→AGENDA→APPLY_AGENDA→EVALUATE→HOMEWORK→CLOSING→ENDED.
- In APPLY_AGENDA, set objective_disposition=completed for addressed work,
  deferred for intentionally postponed work, or not_addressed for unfinished work.
  Disposition applies only to the current objective and only when selected_action
  is transition. For other actions, set objective_disposition=none. On transition,
  remaining objectives are deferred unless the patient explicitly declines them.
- Do not select intervene outside APPLY_AGENDA. In CHECK_IN_REVIEW, gather only
  enough information to meet its exit criterion; do not continue problem work once it is met.
- In AGENDA, use transition once the patient and therapist agree on the focus.
- In CLOSING, use transition to end the session.

Return only: selected_action, goal, intervention_id, intervention_control, objective_disposition.

Do not write the patient-facing response. Keep guidance concise.
"""
)

# Therapist Agent

In [15]:
therapist_agent = create_agent(
    model=ChatGoogleGenerativeAI(
        model="gemma-4-26b-a4b-it",
        temperature=0.6,
        thinking_level="minimal"),
    tools=[],
    system_prompt="""
    You are the conversational therapist component of a larger mental health agentic system.
    You are being Guided by a therapy decision agent.

    Response Rules:
    - Respond in a natural, empathetic, emotionally aware, and respectful way.
    - Generally respond in a clear, and simple way.
    - Allow appropriate humor and playful responses when they fit the patient's tone and the conversation context.
    - Minimize repetitive openings like "that sounds like" or "I am sorry to hear" and vary responses naturally rather than following a fixed pattern
    - Do not default to acknowledging feelings followed by a question. Respond directly, or ask a question depending on what the patient needs.
    - Use the full session transcript as memory. Accept the patient's corrections and build on what they have already told you.
    - Do not repeat a question, suggestion, or interpretation the patient has answered or corrected. If the answer is already clear, respond to it and move forward.
    - Do not assume the patient is feeling bad or experiencing distress without evidence.
    - Ask at most one question, only when useful.
    - Prefer 1-2 short sentences, usually under 30 words; use more only if  necessary.
    - Try to push the conversation forward, don't merely paraphrase what the patient just said.

    Session Rules:
    - Focus on what the patient said while also following the current phase, and objectives.
    - Use INTERVENTION GUIDANCE only when it is provided; otherwise follow the selected conversational action without starting a formal exercise.
    - If an intervention is paused, do not continue or resume it unless the reasoning output explicitly selected an active intervention.
    - When a patient mentions suicidal thoughts, respond to their concerns with support and care and continue therapy. 
    - Just a mention of suicide or patient being suicidal does not mean its an emergency. 
    
    Output:
    A concise response to the user.
"""
)

# sim


In [16]:
def get_message_text(content):
    """Extract text from a model response."""
    if isinstance(content, str):
        return content

    if isinstance(content, list):
        return "\n".join(
            block["text"]
            for block in content
            if isinstance(block, dict)
            and block.get("type") == "text"
        )

    return str(content)

In [17]:
PHASES = ["CHECK_IN_REVIEW", "AGENDA", "APPLY_AGENDA", "EVALUATE", "HOMEWORK", "CLOSING"]
PHASE_GOALS = {
    "CHECK_IN_REVIEW": "Check how the patient is doing and review relevant recent progress.",
    "AGENDA": "Collaboratively agree on the session focus.",
    "APPLY_AGENDA": "Work on the current planned objective, with optional agreed interventions.",
    "EVALUATE": "Review progress, fit, and next steps.",
    "HOMEWORK": "Discuss an optional next step or homework.",
    "CLOSING": "Summarize and bring the session to a close."
}
PHASE_EXIT_CRITERIA = {
    "CHECK_IN_REVIEW": "Current status and relevant recent progress are sufficiently understood to set the agenda.",
    "AGENDA": "The patient and therapist have agreed on a realistic focus for this session.",
    "APPLY_AGENDA": "All objectives are completed or explicitly deferred; the controller records any remaining objectives as deferred on transition.",
    "EVALUATE": "The usefulness, progress, and any needed adjustment have been reviewed.",
    "HOMEWORK": "Any next step is clear, feasible, and collaboratively agreed, or the patient has declined one.",
    "CLOSING": "The session has been summarized and appropriately brought to a close.",
}

session_state = {
    "current_phase": PHASES[0],
    "status": "ACTIVE",
    "objective_index": 0,
    "active_intervention": None,
    "intervention_history": [],
    "objective_results": {},
}
sessionHistory = "No previous turns."

def current_objective():
    i = session_state["objective_index"]
    return session_objectives[i] if i < len(session_objectives) else None

def capture_turn_context():
    phase = session_state["current_phase"]
    active = session_state["active_intervention"]
    return {
        "phase": phase,
        "objective": current_objective() if phase == "APPLY_AGENDA" else None,
        "active_intervention": dict(active) if active else None,
        "intervention_history": list(session_state["intervention_history"]),
    }

def guidance_for_turn(decision, context):
    intervention_id = decision.intervention_id
    if decision.selected_action != "intervene" or intervention_id not in planner_interventions:
        return None
    active = context["active_intervention"]
    if decision.intervention_control == "pause":
        return None
    if active and active["status"] == "PAUSED" and active["id"] == intervention_id:
        if decision.intervention_control != "resume":
            return None
    elif decision.intervention_control == "resume":
        return None
    return delivery_guidance(INTERVENTIONS_BY_ID[intervention_id])

def log_intervention_event(intervention, event, reason=None):
    session_state["intervention_history"].append({
        "intervention_id": intervention["id"],
        "event": event,
        "reason": reason,
    })

def finish_intervention(outcome):
    active = session_state["active_intervention"]
    if active:
        log_intervention_event(active, outcome, active.get("pause_reason"))
        session_state["active_intervention"] = None

def advance_phase():
    if session_state["current_phase"] == "CLOSING":
        session_state["status"] = "ENDED"
    else:
        index = PHASES.index(session_state["current_phase"])
        session_state["current_phase"] = PHASES[index + 1]

def update_session(decision):
    """Apply the decision agent's intervention control and track session progress."""
    phase = session_state["current_phase"]
    active = session_state["active_intervention"]

    if (phase == "APPLY_AGENDA" and decision.selected_action == "transition"
            and decision.objective_disposition != "none" and current_objective()):
        session_state["objective_results"][session_state["objective_index"]] = decision.objective_disposition.upper()
        if decision.objective_disposition in ("completed", "deferred"):
            session_state["objective_index"] += 1

    if decision.selected_action == "transition":
        if phase == "APPLY_AGENDA":
            while current_objective():
                session_state["objective_results"][session_state["objective_index"]] = "DEFERRED"
                session_state["objective_index"] += 1
            finish_intervention("ENDED_WITH_PHASE")
        advance_phase()
        return

    if (decision.intervention_control == "pause" and active
            and active["status"] == "ACTIVE" and decision.intervention_id == active["id"]):
        active["status"] = "PAUSED"
        active["pause_reason"] = decision.goal
        log_intervention_event(active, "PAUSED", decision.goal)

    if phase == "APPLY_AGENDA" and decision.selected_action == "intervene":
        intervention_id = decision.intervention_id
        if intervention_id in planner_interventions:
            if active is None and decision.intervention_control == "none":
                session_state["active_intervention"] = {"id": intervention_id, "status": "ACTIVE", "pause_reason": None}
                log_intervention_event(session_state["active_intervention"], "STARTED")
            elif active["id"] == intervention_id and active["status"] == "PAUSED":
                if decision.intervention_control == "resume":
                    active["status"] = "ACTIVE"
                    active["pause_reason"] = None
                    log_intervention_event(active, "RESUMED")
            elif active["id"] != intervention_id and decision.intervention_control == "none":
                finish_intervention("CHANGED")
                session_state["active_intervention"] = {"id": intervention_id, "status": "ACTIVE", "pause_reason": None}
                log_intervention_event(session_state["active_intervention"], "STARTED")



In [18]:
while session_state["status"] == "ACTIVE":
    patientQuery = input("\nPatient: ").strip()
    if patientQuery.lower() == "exit":
        print("Simulation stopped by user.")
        break
    if not patientQuery:
        continue

    recentHistory = sessionHistory
    turn_context = capture_turn_context()
    phase = turn_context["phase"]
    phase_interventions = approved_intervention_metadata if phase == "APPLY_AGENDA" else []
    reasoning_result = reasoning_agent.invoke({"messages": [{
        "role": "user",
        "content": f"""
PATIENT STATE: {json.dumps(patientState, ensure_ascii=False, default=str)}
SESSION PLAN: {Session_plan.model_dump_json()}
CURRENT PHASE: {phase} — {PHASE_GOALS[phase]}
PHASE EXIT CRITERION: {PHASE_EXIT_CRITERIA[phase]}
CURRENT OBJECTIVE: {turn_context['objective']}
APPROVED INTERVENTIONS: {json.dumps(phase_interventions, ensure_ascii=False)}
ACTIVE INTERVENTION: {turn_context['active_intervention']}
INTERVENTION HISTORY: {json.dumps(turn_context['intervention_history'])}
RECENT HISTORY: {recentHistory}
PATIENT: {patientQuery}
"""
    }]})
    decision = reasoning_result["structured_response"]
    print("\n--- Therapy Decision ---")
    print(decision.model_dump_json(indent=2))

    intervention_guidance = guidance_for_turn(decision, turn_context)
    therapist_result = therapist_agent.invoke({"messages": [{
        "role": "user",
        "content": f"""
PATIENT STATE: {json.dumps(patientState, ensure_ascii=False, default=str)}
PATIENT QUERY: {patientQuery}
RECENT SESSION HISTORY: {recentHistory}
SESSION PLAN: {Session_plan.model_dump_json()}
CURRENT PHASE: {phase} — {PHASE_GOALS[phase]}
CURRENT OBJECTIVE: {turn_context['objective']}
THERAPY REASONING AGENT OUTPUT: {decision.model_dump_json()}
INTERVENTION GUIDANCE: {json.dumps(intervention_guidance, ensure_ascii=False)}
"""
    }]})
    therapist_response = get_message_text(therapist_result["messages"][-1].content).strip()
    print("\n--- Therapist ---")
    print(therapist_response)
    sessionHistory += f"\nPatient: {patientQuery}\nTherapist: {therapist_response}\n"
    update_session(decision)
    print("\n--- Session State ---")
    print(json.dumps(session_state, indent=2))


Patient:  Good morning doctor



--- Therapy Decision ---
{
  "selected_action": "answer",
  "goal": "Acknowledge the greeting and initiate the session start.",
  "intervention_id": null,
  "intervention_control": "none",
  "objective_disposition": "none"
}

--- Therapist ---
Good morning! How have things been going for you since we last connected?

--- Session State ---
{
  "current_phase": "CHECK_IN_REVIEW",
  "status": "ACTIVE",
  "objective_index": 0,
  "active_intervention": null,
  "intervention_history": [],
  "objective_results": {}
}



Patient:  It's been going pretty decent, I guess, it's a mixed bag. I've been focusing on work and that's been going well, but I'm still struggling to gather the energy for school and getting out of bed for my personal life.



--- Therapy Decision ---
{
  "selected_action": "explore",
  "goal": "Understand the specific challenges and context regarding the patient's energy levels and difficulty getting out of bed.",
  "intervention_id": null,
  "intervention_control": "none",
  "objective_disposition": "none"
}

--- Therapist ---
It sounds like there's a bit of a tug-of-war between your work productivity and your energy for everything else. When you're struggling to get out of bed, what does that moment usually feel like for you?

--- Session State ---
{
  "current_phase": "CHECK_IN_REVIEW",
  "status": "ACTIVE",
  "objective_index": 0,
  "active_intervention": null,
  "intervention_history": [],
  "objective_results": {}
}



Patient:  If I had to describe it, like a weight on my shoulders. Knowing that getting out of bed means a bunch of tasks, it overwhelms me a bit.



--- Therapy Decision ---
{
  "selected_action": "explore",
  "goal": "Understand the physical and emotional experience of being overwhelmed by tasks to better understand the barriers to morning routine.",
  "intervention_id": null,
  "intervention_control": "none",
  "objective_disposition": "none"
}

--- Therapist ---
That heavy sensation makes sense when you're looking at a mountain of tasks all at once. Does that weight feel more like physical exhaustion or more like mental dread?

--- Session State ---
{
  "current_phase": "CHECK_IN_REVIEW",
  "status": "ACTIVE",
  "objective_index": 0,
  "active_intervention": null,
  "intervention_history": [],
  "objective_results": {}
}


... (79 outputs left out)


--- Therapy Decision ---
{
  "selected_action": "reframe",
  "goal": "Acknowledge the patient's perspective on the feeling of being stuck and provide a grounded alternative perspective to help reduce self-judgment.",
  "intervention_id": null,
  "intervention_control": "none",
  "objective_disposition": "none"
}

--- Therapist ---
It makes sense that it feels like a failure when the momentum breaks, especially when you're striving for that sense of mastery. 

Could we try reframing that "stuck" moment not as a failure, but as a natural signal that you've reached the edge of your current knowledge and need a new tool?

--- Session State ---
{
  "current_phase": "APPLY_AGENDA",
  "status": "ACTIVE",
  "objective_index": 0,
  "active_intervention": {
    "id": "ba_activity_scheduling_and_structuring",
    "status": "ACTIVE",
    "pause_reason": null
  },
  "intervention_history": [
    {
      "intervention_id": "ba_activity_scheduling_and_structuring",
      "event": "STARTED",
      "r


Patient:  That's one way to see it, I hope I can see it that way too



--- Therapy Decision ---
{
  "selected_action": "validate",
  "goal": "Acknowledge the patient's skepticism or hesitation about the reframe while maintaining the validity of the perspective.",
  "intervention_id": null,
  "intervention_control": "none",
  "objective_disposition": "none"
}


GoogleRateLimitError: Error calling model 'gemma-4-26b-a4b-it' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 16000, model: gemma-4-26b, description: Generate content input tokens per minute limit exceeded on Free Tier. See https://ai.google.dev/gemini-api/docs/rate-limits.\nPlease retry in 9.106317437s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'descr